# MPI — Guided debugging workshop

Start with two hello-world examples, then fix seven buggy C programs.
Basic familiarity with C variables, loops and arrays is sufficient;
MPI concepts are introduced through the examples and assignments.

Run cells **individually**, not with Run All. For each debugging activity:

1. Inspect the source and predict what it does.
2. Run it and observe the result.
3. Edit and save your own `exercises/codeN/code.c` in JupyterLab.
4. Rerun it, then optionally compare with the reference.

This notebook never applies corrections. Viewing a reference never
changes your working source. Keep your corrections and notes.

Use only organiser-authorised participant resources. Some activities
require four local MPI processes: ask the organiser before running them
if your allocation does not permit this. Do not launch these activities
on a cluster head node or outside the approved participant environment.

## 1. Check the prepared environment

Open this notebook with its working directory set to `HPC/MPI/`.
The next cell checks the existing helper and MPI tool paths, then adds
those directories to this notebook process's PATH. It installs nothing,
compiles nothing and launches no MPI processes.

This PATH change affects commands launched by this notebook, not
separately opened terminals. The preparation receipt and executable-file
checks do **not** prove MPI runtime readiness.

If a check fails, ask the organiser. Do not reinstall or reset the
shared OpenMP/MPI environment.

In [ ]:
import os
from pathlib import Path
import shutil

module_root = Path.cwd().resolve()
helper_directory = module_root / "helpers"
helper = helper_directory / "mpi_start"

if not helper.is_file() or not (module_root / "exercises").is_dir():
    raise RuntimeError("Open this notebook with its working directory set to HPC/MPI/.")
if not os.access(helper, os.X_OK):
    raise RuntimeError("The MPI helper is not executable; ask the organiser.")

hpc_prefix = Path.home() / ".local/share/digitafrica/runtime/hpc/environment"
receipt = hpc_prefix.parent / "environment.sha256"
if not receipt.is_file():
    raise RuntimeError("Prepared HPC receipt is missing; ask the organiser.")

for name in ("mpicc", "mpirun"):
    tool = hpc_prefix / "bin" / name
    if not tool.is_file() or not os.access(tool, os.X_OK):
        raise RuntimeError(f"Prepared MPI tool is unavailable: {tool}; ask the organiser.")

existing = os.environ.get("PATH", "").split(os.pathsep)
preferred = [str(helper_directory), str(hpc_prefix / "bin")]
os.environ["PATH"] = os.pathsep.join(preferred + [p for p in existing if p not in preferred])

print("Module directory:", module_root)
print("MPI compiler:", hpc_prefix / "bin/mpicc")
print("MPI launcher:", hpc_prefix / "bin/mpirun")
print("Exercise helper:", shutil.which("mpi_start"))
print("No software was installed and no exercise source was changed.")
print("Tool-file checks passed; MPI execution has not yet been tested.")

In [ ]:
!mpi_start --help

## 2. Understand the commands

- `mpi_start code1`: display your current C source.
- `mpi_start code1 --problem`: display the assignment.
- `mpi_start code1 --run --processes 2`: compile and run your current source.
- `mpi_start code1 --solution`: display reference code and explanation.

`--timeout` limits each execution in seconds; `--repeat` repeats execution.
The helper rebuilds into a temporary directory and never runs an old
executable after a failed build. It runs locally and does not submit
Slurm jobs or request remote hosts.

A timeout, MPI error or wrong result can be the intended observation.
Invalid MPI programs may behave differently across implementations.
A successful exit is not proof of correct communication or calculation.

The example commands request either two or four processes. Do not
increase resource use beyond the organiser's instructions.

## 3. Hello-world examples

MPI processes have separate address spaces. Each process has a rank in a
communicator, and `MPI_Comm_size` reports that communicator's process count.

These examples need no bug correction. The second example uses rank-based
branches within the same program: this is still SPMD, not evidence that
separate programs are being launched. Output order is not guaranteed.

## MPI hello world: processes and ranks

### Observe the example

Run the example. Identify the rank and total process count in its output. No correction is required.

1. Inspect the source and predict its behaviour:

       mpi_start hello1

2. Run it in the organiser-authorised participant environment:

       mpi_start hello1 --run --processes 2 --timeout 10

3. Record what you observe.

### Check

With two processes, expect a greeting from ranks 0 and 1. Output order and line interleaving are not guaranteed.

A timeout, MPI error or wrong result can be part of a debugging exercise.
A successful exit alone does not establish correctness. Invalid MPI code
may behave differently across implementations.

### Reference

For these introductory examples, the reference is the same program. Viewing never overwrites your source.

    mpi_start hello1 --solution

Explain the rank-dependent output.

Use only authorised resources. Do not launch MPI on a cluster head node,
install software, or reset the shared OpenMP/MPI environment.

### Attribution

The C source retains its original author header. See `../../ADAPTATION.md`
for source revision and changes. This activity text is newly authored,
distributed under Apache-2.0; see `../../../LICENSE`.

In [ ]:
!mpi_start hello1

In [ ]:
!mpi_start hello1 --run --processes 2 --timeout 10

Record which ranks print which messages. Different output ordering is not a bug.

## Different behaviour by rank

### Observe the example

Run the example. Identify which branch each rank executes. No correction is required. This remains a single-program (SPMD) example, even though ranks take different branches.

1. Inspect the source and predict its behaviour:

       mpi_start hello2

2. Run it in the organiser-authorised participant environment:

       mpi_start hello2 --run --processes 4 --timeout 10

3. Record what you observe.

### Check

With four processes, rank 0 identifies itself, rank 1 prints Hello world, rank 2 prints Bye world, and rank 3 has no greeting. Output order is not guaranteed.

A timeout, MPI error or wrong result can be part of a debugging exercise.
A successful exit alone does not establish correctness. Invalid MPI code
may behave differently across implementations.

### Reference

For these introductory examples, the reference is the same program. Viewing never overwrites your source.

    mpi_start hello2 --solution

Explain the rank-dependent output.

Use only authorised resources. Do not launch MPI on a cluster head node,
install software, or reset the shared OpenMP/MPI environment.

### Attribution

The C source retains its original author header. See `../../ADAPTATION.md`
for source revision and changes. This activity text is newly authored,
distributed under Apache-2.0; see `../../../LICENSE`.

In [ ]:
!mpi_start hello2

In [ ]:
!mpi_start hello2 --run --processes 4 --timeout 10

Record which ranks print which messages. Different output ordering is not a bug.

## 4. Fix the buggy programs

Each activity has one focused correction task. The distributed versions
already use short workloads and, where necessary, documented baseline
safety fixes. Original source attribution and adaptation details are in
`ADAPTATION.md`.

The first execution uses the buggy working source. After editing, the
second execution uses your saved correction—not the reference file.

## A message exchange that does not finish

### Fix the buggy code

The two-process exchange does not finish. Check the source, destination and tag of each send and receive. Fix the mismatch.

1. Inspect the source and predict its behaviour:

       mpi_start code1

2. Run it in the organiser-authorised participant environment:

       mpi_start code1 --run --processes 2 --timeout 10

3. Edit and save `exercises/code1/code.c`, then rerun the command.

### Check

Both ranks report sending and receiving one character, and the program finishes.

A timeout, MPI error or wrong result can be part of a debugging exercise.
A successful exit alone does not establish correctness. Invalid MPI code
may behave differently across implementations.

### Reference

Only after your own correction, compare with the reference. Viewing never overwrites your source.

    mpi_start code1 --solution

Briefly record the bug, your change and the evidence that it works.

Use only authorised resources. Do not launch MPI on a cluster head node,
install software, or reset the shared OpenMP/MPI environment.

### Attribution

The C source retains its original author header. See `../../ADAPTATION.md`
for source revision and changes. This activity text is newly authored,
distributed under Apache-2.0; see `../../../LICENSE`.

In [ ]:
!mpi_start code1

In [ ]:
!mpi_start code1 --run --processes 2 --timeout 10

### Make your correction

Open **`exercises/code1/code.c`** in the JupyterLab text editor.
Make your correction and save it. Then run the next cell to rebuild
and test your working source. Check the expected behaviour in the assignment.

In [ ]:
!mpi_start code1 --run --processes 2 --timeout 10

### Optional reference — after your attempt

Skip the next cell until you have made and tested your own correction.
It displays the reference and explanation without changing your file.

In [ ]:
!mpi_start code1 --solution

Briefly record the bug, your change and the evidence that it works.

## Unexpected received values

### Fix the buggy code

The received values do not reliably match the sent integers. Check both the C variable types and the MPI datatypes. Fix the mismatch.

1. Inspect the source and predict its behaviour:

       mpi_start code2

2. Run it in the organiser-authorised participant environment:

       mpi_start code2 --run --processes 2 --timeout 10

3. Edit and save `exercises/code2/code.c`, then rerun the command.

### Check

Rank 1 receives the integers 0, 10, 20, ..., 90 in order. Output from different ranks may interleave.

A timeout, MPI error or wrong result can be part of a debugging exercise.
A successful exit alone does not establish correctness. Invalid MPI code
may behave differently across implementations.

### Reference

Only after your own correction, compare with the reference. Viewing never overwrites your source.

    mpi_start code2 --solution

Briefly record the bug, your change and the evidence that it works.

Use only authorised resources. Do not launch MPI on a cluster head node,
install software, or reset the shared OpenMP/MPI environment.

### Attribution

The C source retains its original author header. See `../../ADAPTATION.md`
for source revision and changes. This activity text is newly authored,
distributed under Apache-2.0; see `../../../LICENSE`.

In [ ]:
!mpi_start code2

In [ ]:
!mpi_start code2 --run --processes 2 --timeout 10

### Make your correction

Open **`exercises/code2/code.c`** in the JupyterLab text editor.
Make your correction and save it. Then run the next cell to rebuild
and test your working source. Check the expected behaviour in the assignment.

In [ ]:
!mpi_start code2 --run --processes 2 --timeout 10

### Optional reference — after your attempt

Skip the next cell until you have made and tested your own correction.
It displays the reference and explanation without changing your file.

In [ ]:
!mpi_start code2 --solution

Briefly record the bug, your change and the evidence that it works.

## Missing MPI initialisation and shutdown

### Fix the buggy code

Compare the MPI lifecycle with hello1. Add the missing initialisation and shutdown calls in the correct places.

1. Inspect the source and predict its behaviour:

       mpi_start code3

2. Run it in the organiser-authorised participant environment:

       mpi_start code3 --run --processes 4 --timeout 10

3. Edit and save `exercises/code3/code.c`, then rerun the command.

### Check

The program starts MPI, processes the array, prints sample results and a final sum, and finishes. The mathematical total is 255984000; single-precision accumulation can introduce rounding differences. Finishing alone does not verify the numerical result.

A timeout, MPI error or wrong result can be part of a debugging exercise.
A successful exit alone does not establish correctness. Invalid MPI code
may behave differently across implementations.

### Reference

Only after your own correction, compare with the reference. Viewing never overwrites your source.

    mpi_start code3 --solution

Briefly record the bug, your change and the evidence that it works.

Use only authorised resources. Do not launch MPI on a cluster head node,
install software, or reset the shared OpenMP/MPI environment.

### Attribution

The C source retains its original author header. See `../../ADAPTATION.md`
for source revision and changes. This activity text is newly authored,
distributed under Apache-2.0; see `../../../LICENSE`.

In [ ]:
!mpi_start code3

In [ ]:
!mpi_start code3 --run --processes 4 --timeout 10

### Make your correction

Open **`exercises/code3/code.c`** in the JupyterLab text editor.
Make your correction and save it. Then run the next cell to rebuild
and test your working source. Check the expected behaviour in the assignment.

In [ ]:
!mpi_start code3 --run --processes 4 --timeout 10

### Optional reference — after your attempt

Skip the next cell until you have made and tested your own correction.
It displays the reference and explanation without changing your file.

In [ ]:
!mpi_start code3 --solution

Briefly record the bug, your change and the evidence that it works.

## An incomplete collective reduction

### Fix the buggy code

Check which ranks call MPI_Reduce. Make every rank participate in the matching reduction before the root prints its final sum.

1. Inspect the source and predict its behaviour:

       mpi_start code4

2. Run it in the organiser-authorised participant environment:

       mpi_start code4 --run --processes 4 --timeout 10

3. Edit and save `exercises/code4/code.c`, then rerun the command.

### Check

With the workshop array size of 20000, the final sum is 399980000. The root must print the reduced updated total, not the initial array sum.

A timeout, MPI error or wrong result can be part of a debugging exercise.
A successful exit alone does not establish correctness. Invalid MPI code
may behave differently across implementations.

### Reference

Only after your own correction, compare with the reference. Viewing never overwrites your source.

    mpi_start code4 --solution

Briefly record the bug, your change and the evidence that it works.

Use only authorised resources. Do not launch MPI on a cluster head node,
install software, or reset the shared OpenMP/MPI environment.

### Attribution

The C source retains its original author header. See `../../ADAPTATION.md`
for source revision and changes. This activity text is newly authored,
distributed under Apache-2.0; see `../../../LICENSE`.

In [ ]:
!mpi_start code4

In [ ]:
!mpi_start code4 --run --processes 4 --timeout 10

### Make your correction

Open **`exercises/code4/code.c`** in the JupyterLab text editor.
Make your correction and save it. Then run the next cell to rebuild
and test your working source. Check the expected behaviour in the assignment.

In [ ]:
!mpi_start code4 --run --processes 4 --timeout 10

### Optional reference — after your attempt

Skip the next cell until you have made and tested your own correction.
It displays the reference and explanation without changing your file.

In [ ]:
!mpi_start code4 --solution

Briefly record the bug, your change and the evidence that it works.

## A message exchange that never stops

### Fix the buggy code

Make both ranks exchange exactly 20 messages, then reach MPI_Finalize. Keep the receiver's small work loop. Use a separate counter for the communication loop.

1. Inspect the source and predict its behaviour:

       mpi_start code5

2. Run it in the organiser-authorised participant environment:

       mpi_start code5 --run --processes 2 --timeout 10

3. Edit and save `exercises/code5/code.c`, then rerun the command.

### Check

Both ranks report completing 20 messages and the program finishes. The original infinite version is expected to reach the helper timeout.

A timeout, MPI error or wrong result can be part of a debugging exercise.
A successful exit alone does not establish correctness. Invalid MPI code
may behave differently across implementations.

### Reference

Only after your own correction, compare with the reference. Viewing never overwrites your source.

    mpi_start code5 --solution

Briefly record the bug, your change and the evidence that it works.

Use only authorised resources. Do not launch MPI on a cluster head node,
install software, or reset the shared OpenMP/MPI environment.

### Attribution

The C source retains its original author header. See `../../ADAPTATION.md`
for source revision and changes. This activity text is newly authored,
distributed under Apache-2.0; see `../../../LICENSE`.

In [ ]:
!mpi_start code5

In [ ]:
!mpi_start code5 --run --processes 2 --timeout 10

### Make your correction

Open **`exercises/code5/code.c`** in the JupyterLab text editor.
Make your correction and save it. Then run the next cell to rebuild
and test your working source. Check the expected behaviour in the assignment.

In [ ]:
!mpi_start code5 --run --processes 2 --timeout 10

### Optional reference — after your attempt

Skip the next cell until you have made and tested your own correction.
It displays the reference and explanation without changing your file.

In [ ]:
!mpi_start code5 --solution

Briefly record the bug, your change and the evidence that it works.

## Waiting for requests that were not created

### Fix the buggy code

Check which calls create MPI_Request handles. Correct the number of requests rank 2 passes to MPI_Waitall.

1. Inspect the source and predict its behaviour:

       mpi_start code6

2. Run it in the organiser-authorised participant environment:

       mpi_start code6 --run --processes 4 --timeout 10

3. Edit and save `exercises/code6/code.c`, then rerun the command.

### Check

All four ranks complete their communication and report timings. Timing differences are not a correctness test; inspect request creation and completion as well.

A timeout, MPI error or wrong result can be part of a debugging exercise.
A successful exit alone does not establish correctness. Invalid MPI code
may behave differently across implementations.

### Reference

Only after your own correction, compare with the reference. Viewing never overwrites your source.

    mpi_start code6 --solution

Briefly record the bug, your change and the evidence that it works.

Use only authorised resources. Do not launch MPI on a cluster head node,
install software, or reset the shared OpenMP/MPI environment.

### Attribution

The C source retains its original author header. See `../../ADAPTATION.md`
for source revision and changes. This activity text is newly authored,
distributed under Apache-2.0; see `../../../LICENSE`.

In [ ]:
!mpi_start code6

In [ ]:
!mpi_start code6 --run --processes 4 --timeout 10

### Make your correction

Open **`exercises/code6/code.c`** in the JupyterLab text editor.
Make your correction and save it. Then run the next cell to rebuild
and test your working source. Check the expected behaviour in the assignment.

In [ ]:
!mpi_start code6 --run --processes 4 --timeout 10

### Optional reference — after your attempt

Skip the next cell until you have made and tested your own correction.
It displays the reference and explanation without changing your file.

In [ ]:
!mpi_start code6 --solution

Briefly record the bug, your change and the evidence that it works.

## An inconsistent broadcast

### Fix the buggy code

Broadcast one integer from rank 0. Check the count passed by each rank and the size of its receive buffer. Correct the count.

1. Inspect the source and predict its behaviour:

       mpi_start code7

2. Run it in the organiser-authorised participant environment:

       mpi_start code7 --run --processes 2 --timeout 10

3. Edit and save `exercises/code7/code.c`, then rerun the command.

### Check

Every rank prints value 23 after the broadcast. Before the broadcast, only rank 0 starts with 23.

A timeout, MPI error or wrong result can be part of a debugging exercise.
A successful exit alone does not establish correctness. Invalid MPI code
may behave differently across implementations.

### Reference

Only after your own correction, compare with the reference. Viewing never overwrites your source.

    mpi_start code7 --solution

Briefly record the bug, your change and the evidence that it works.

Use only authorised resources. Do not launch MPI on a cluster head node,
install software, or reset the shared OpenMP/MPI environment.

### Attribution

The C source retains its original author header. See `../../ADAPTATION.md`
for source revision and changes. This activity text is newly authored,
distributed under Apache-2.0; see `../../../LICENSE`.

In [ ]:
!mpi_start code7

In [ ]:
!mpi_start code7 --run --processes 2 --timeout 10

### Make your correction

Open **`exercises/code7/code.c`** in the JupyterLab text editor.
Make your correction and save it. Then run the next cell to rebuild
and test your working source. Check the expected behaviour in the assignment.

In [ ]:
!mpi_start code7 --run --processes 2 --timeout 10

### Optional reference — after your attempt

Skip the next cell until you have made and tested your own correction.
It displays the reference and explanation without changing your file.

In [ ]:
!mpi_start code7 --solution

Briefly record the bug, your change and the evidence that it works.

## Final review

For each corrected program, identify the communication or lifecycle rule
that was violated. Distinguish a successful exit from evidence of the
intended result. Keep your saved corrections and notes.

OpenMP and MPI share the prepared environment. No reset, reinstall or
runtime transition is needed; wait for the organiser's next instruction.

## Attribution and licence

The C programs are adapted from existing Tutorials materials and retain
their original author headers. See `ADAPTATION.md`, `source-manifest.json`
and `../PROVENANCE.md` for the applicable source and adaptation records.
This newly authored notebook was created with AI assistance and is
provided under Apache-2.0; see `../LICENSE`. Retained third-party source
notices are not replaced by the notebook's licence statement.